# 24. AMP — Automatic Mixed Precision

Источник (ментор, **не** в видео Дубинина):
[AMP examples — PyTorch 2.15](https://docs.pytorch.org/docs/2.15/notes/amp_examples.html)

## Зачем эта тема одной фразой

Обычное обучение почти всё считает в **float32** (точные числа). На большой модели (Transformer и т.п.) это:
- съедает много **памяти GPU**;
- часто **медленнее**, чем могло бы.

**AMP** = «где можно — считай потоньше (`float16` / `bfloat16`), где опасно — оставь float32».  
На практике это два инструмента:

1. **`autocast`** — PyTorch сам выбирает dtype для операций внутри блока `with`.
2. **`GradScaler`** — временно увеличивает loss/градиенты, чтобы мелкие fp16-градиенты не обнулились, потом возвращает нормальный масштаб.

> **Обзор:** [`24_amp_обзор.md`](24_amp_обзор.md) · конспект [`d.4.2` §2.8](../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.2 Градиентная оптимизация.md)

| блок | что поймёшь |
|:---|:---|
| 0 | настройка device / нужен ли scaler на твоей машине |
| 1 | что делают `GradScaler` и `autocast` по отдельности |
| 2 | полный train-step «как в доке» |
| 3 | как правильно клипать градиенты вместе с AMP |
| 4 | как копить градиенты (accumulation) с AMP |

**Как читать.** Под каждым блоком — зачем он нужен; в коде — комментарии «что делает строка».  
Прогоняй ячейки сверху вниз. Дубликат с решениями: `24_amp_решения.ipynb`.

**Важно про твой ПК:** часто **нет CUDA**. Тогда ускорения почти нет, но **API тот же**. Мы пишем цикл как на GPU: на CPU scaler обычно `enabled=False` (no-op), dtype — `bfloat16`.


## 0. Подготовка ячейки

Эта ячейка не про «магию AMP», а про **где** мы считаем и **какие** настройки выбрать.

- `DEVICE` — `"cuda"`, если есть GPU, иначе `"cpu"`.
- `USE_SCALER` — `True` только на CUDA: `GradScaler` нужен в первую очередь против **underflow в float16**. На CPU для практики часто выключаем (`enabled=False`), иначе смысла мало.
- `AMP_DTYPE` — на CUDA обычно `float16`, на CPU удобнее `bfloat16` (шире диапазон, underflow реже).

`need(...)` — наша assert-проверка, как в прошлых тренажёрах.


In [ ]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# GradScaler критичен для float16 (мелкие градиенты → 0). На CUDA включаем.
USE_SCALER = DEVICE.type == "cuda"
# На CUDA типичный AMP dtype = float16; на CPU берём bfloat16.
AMP_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.bfloat16

print("torch", torch.__version__)
print("device", DEVICE, "| scaler", USE_SCALER, "| amp dtype", AMP_DTYPE)

## 1. Два инструмента AMP

Представь обычный шаг обучения:

`forward → loss → backward → optimizer.step`

AMP встраивается так:

| кусок | инструмент | зачем |
|:---|:---|:---|
| forward + loss | `autocast` | часть матричных умножений идёт в «тонком» dtype → быстрее/легче |
| backward + step | `GradScaler` | не дать мелким fp16-градиентам стать нулём |

**Веса модели** (`nn.Linear.weight` и т.д.) по умолчанию остаются **float32**.  
Меняется dtype **активаций/промежуточных вычислений** внутри `autocast`, а не «вся сеть навсегда в fp16».

`GradScaler(device, enabled=...)`:
- создаём **один раз** на всё обучение (не каждый батч);
- `enabled=False` → ведёт себя как «прозрачный» слой: `scale/step/update` можно писать одинаково на CPU и GPU.


In [ ]:
# 1.1 GradScaler — объект на всё обучение
# Зачем: один раз создать «масштабировщик» градиентов.
# DEVICE.type: 'cuda' или 'cpu' — к какому устройству привязан scaler.
# enabled=USE_SCALER: на CPU у нас False → API тот же, но реального scaling нет.

scaler = GradScaler(DEVICE.type, enabled=USE_SCALER)

need(isinstance(scaler, GradScaler), "torch.amp.GradScaler")
need(scaler.is_enabled() == USE_SCALER, "enabled согласован с USE_SCALER")
print("1.1 — верно", "enabled=", scaler.is_enabled())

### 1.2 Что делает `with autocast(...)`

```python
with autocast(device_type=..., dtype=...):
    y = lin(xb)
```

Пока мы **внутри** `with`:
- PyTorch для подходящих операций может посчитать результат в `AMP_DTYPE` (`bfloat16` / `float16`);
- это **не** переписывает веса слоя навсегда — только этот forward.

Снаружи `with` тот же `lin(xb)` снова даёт обычный **float32** (если вход float32).

Зачем смотреть dtype выхода: чтобы увидеть глазами — «autocast реально переключил точность», а не просто декоративный `with`.


In [ ]:
# 1.2 autocast: внутри блока dtype может стать «тонким»
lin = nn.Linear(4, 3).to(DEVICE)  # веса Linear по умолчанию float32
xb = torch.randn(8, 4, device=DEVICE)  # вход float32

# device_type — где считаем; dtype — целевая «тонкая» точность
with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE):
    y = lin(xb)

# Тот же слой вне autocast — снова float32
y32 = lin(xb)

need(y32.dtype == torch.float32, "без autocast float32")
need(
    y.dtype in (AMP_DTYPE, torch.float32),
    f"внутри autocast ожидали {AMP_DTYPE} или fallback float32, got {y.dtype}",
)
print("1.2 — верно", "amp out", y.dtype, "fp32 out", y32.dtype)

## 2. Типичный AMP-шаг (выучить порядок)

Без AMP ты уже писал:

```text
opt.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
opt.step()
```

С AMP почти то же, но две вставки:

```text
opt.zero_grad()
with autocast(...):          # 1) forward+loss в смешанной точности
    pred = model(x)
    loss = loss_fn(pred, y)
scaler.scale(loss).backward()  # 2) backward по УВЕЛИЧЕННОМУ loss
scaler.step(opt)               # сначала unscale градиентов, потом opt.step
                               # (если inf/NaN — step ПРОПУСТИТ)
scaler.update()                # подстроить коэффициент scale на следующий батч
```

**Почему `backward` не кладут внутрь `autocast`:** так рекомендуют доки.  
Backward идёт в тех dtype, которые autocast уже выбрал на forward — отдельно «включать» не надо.

**Что проверяем в задании:** после шага веса реально изменились (иначе AMP-цикл «пустой»).


In [ ]:
# 2.1 один полный AMP train-step
class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))

    def forward(self, x):
        return self.net(x)


def amp_train_step(model, opt, scaler, xb, yb, loss_fn):
    """Один батч: AMP-forward → scaled backward → step → update."""
    # 1) обнулить старые .grad (set_to_none чуть экономнее по памяти)
    opt.zero_grad(set_to_none=True)

    # 2) forward + loss внутри autocast
    with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE):
        pred = model(xb)
        loss = loss_fn(pred, yb)

    # 3) backward по scaled loss (градиенты «крупнее»)
    scaler.scale(loss).backward()

    # 4) unscale внутри step + optimizer.step (или skip при inf/NaN)
    scaler.step(opt)

    # 5) обновить коэффициент scale на следующий батч
    scaler.update()

    return float(loss.detach().item())


torch.manual_seed(0)
model = Tiny().to(DEVICE)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
scaler = GradScaler(DEVICE.type, enabled=USE_SCALER)
loss_fn = nn.MSELoss()
xb = torch.randn(16, 4, device=DEVICE)
yb = torch.randn(16, 2, device=DEVICE)

w_before = model.net[0].weight.detach().clone()
val = amp_train_step(model, opt, scaler, xb, yb, loss_fn)
need(isinstance(val, float), "вернуть float (loss.item())")
need(not torch.equal(w_before, model.net[0].weight.detach()), "веса должны обновиться")
print("2.1 — верно", "loss=", round(val, 4))

## 3. Clipping градиентов + AMP *(дополнение, не ядро AMP)*

**Важно:** большой градиент бывает **не из‑за AMP**. Взрыв градиентов — обычная история и при чистом float32.  
Ядро AMP — это примеры 1–2 (`autocast` + `GradScaler`). Их достаточно для базового цикла: перед `step` scaler сам вернёт нужный размер градиента.

**Clipping** — отдельный приём: если градиент слишком большой, его подрезают `clip_grad_norm_(..., max_norm)`.  
Этот блок только про связку: **если вдруг клипаешь и при этом используешь AMP**, сделай так:

```text
scale(loss).backward()              # градиенты ещё ×scale
scaler.unscale_(optimizer)          # вернуть .grad к «настоящему» масштабу
clip_grad_norm_(model.parameters(), max_norm)  # режем уже честные числа
scaler.step(optimizer)              # знает, что unscale_ уже был
scaler.update()
```

Почему нельзя клипать сразу после `backward`: порог `max_norm` тогда смотрит на увеличенные числа.  
Правило: **`unscale_` один раз** на optimizer за step. Дважды → ошибка.

Если клипать не планируешь — этот блок можно просто знать «на будущее».

**Практика:** сначала учи **без** clip. Если видишь взрывы / `NaN` / дикие скачки loss — тогда добавляй.  
В готовых рецептах больших моделей (часто NLP/Transformer) clip уже бывает в паттерне — там просто копируешь порядок `unscale_ → clip → step`.


In [ ]:
# 3.1 AMP-шаг с clip_grad_norm_
# Отличие от 2.1: между backward и step вставляем unscale_ → clip.


def amp_train_step_clip(model, opt, scaler, xb, yb, loss_fn, max_norm=1.0):
    opt.zero_grad(set_to_none=True)
    with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE):
        pred = model(xb)
        loss = loss_fn(pred, yb)

    scaler.scale(loss).backward()

    # градиенты сейчас scaled → сначала вернуть нормальный масштаб
    scaler.unscale_(opt)
    # теперь max_norm=1.0 означает именно «настоящую» норму градиентов
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

    scaler.step(opt)  # повторно unscale не делает
    scaler.update()
    return float(loss.detach().item())


torch.manual_seed(1)
model = Tiny().to(DEVICE)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
scaler = GradScaler(DEVICE.type, enabled=USE_SCALER)
xb = torch.randn(16, 4, device=DEVICE)
yb = torch.randn(16, 2, device=DEVICE)
w_before = model.net[0].weight.detach().clone()
val = amp_train_step_clip(model, opt, scaler, xb, yb, nn.MSELoss(), max_norm=1.0)
need(isinstance(val, float), "float loss")
need(not torch.equal(w_before, model.net[0].weight.detach()), "веса обновились")
print("3.1 — верно", "loss=", round(val, 4))

## 4. Gradient accumulation + AMP

Иногда батч целиком не влезает в GPU. Тогда делают **K микробатчей**:

- на каждом: `backward` (градиенты **складываются** в `.grad`);
- раз в K шагов: один `optimizer.step`.

С AMP важно:

1. Делить loss на K: `loss = loss / K`, чтобы суммарный градиент был как у большого батча.
2. Пока копим — несколько раз `scaler.scale(loss).backward()`.
3. `scaler.step` + `scaler.update` — **только** когда effective batch собран.
4. `unscale_` (если нужен clip) — тоже только перед этим `step`, не посередине накопления.

Почему нельзя `update` посредине: коэффициент scale изменится, и следующие градиенты добавятся «в другом масштабе» — каша.


In [ ]:
# 4.1 K=2 микробатча → один step
# Схема: zero_grad → (autocast + loss/2 + scale.backward) × 2 → step → update


def amp_accum_two(model, opt, scaler, batch_a, batch_b, loss_fn):
    """batch_* = (xb, yb). Вернуть средний loss двух микробатчей (до деления на K)."""
    opt.zero_grad(set_to_none=True)
    losses = []
    # for идёт по одному микробатчу за круг: сначала batch_a, потом batch_b
    # (не оба сразу). На каждом круге: forward → loss/2 → backward.
    for xb, yb in (batch_a, batch_b):
        with autocast(device_type=DEVICE.type, dtype=AMP_DTYPE):
            pred = model(xb)
            # /2 чтобы сумма двух backward ≈ один большой батч
            loss = loss_fn(pred, yb) / 2
        # backward на этом круге ДОБАВЛЯЕТ градиент к .grad (zero_grad был только до цикла)
        scaler.scale(loss).backward()
        losses.append(float(loss.detach().item()) * 2)  # для лога — «как до /2»

    # effective batch готов → один step + один update
    scaler.step(opt)
    scaler.update()
    return sum(losses) / 2


torch.manual_seed(2)
model = Tiny().to(DEVICE)
opt = torch.optim.SGD(model.parameters(), lr=0.05)
scaler = GradScaler(DEVICE.type, enabled=USE_SCALER)
# два тестовых микробатча (фейковые данные): каждый = (xb: 8×4, yb: 8×2)
microbatch_1 = (torch.randn(8, 4, device=DEVICE), torch.randn(8, 2, device=DEVICE))
microbatch_2 = (torch.randn(8, 4, device=DEVICE), torch.randn(8, 2, device=DEVICE))
w_before = model.net[0].weight.detach().clone()
val = amp_accum_two(model, opt, scaler, microbatch_1, microbatch_2, nn.MSELoss())
need(isinstance(val, float), "float")
need(
    not torch.equal(w_before, model.net[0].weight.detach()),
    "один step после 2 backward",
)
print("4.1 — верно", "mean_loss=", round(val, 4))

## Итог: что забрать из урока

1. **AMP** нужен на больших моделях/GPU: меньше памяти, часто быстрее.
2. **`autocast`** — forward/loss в смешанной точности.
3. **`GradScaler`** — не дать fp16-градиентам обнулиться; перед step возвращает нормальный масштаб.
4. Порядок: `zero_grad → autocast → scale.backward → (unscale+clip) → step → update`.
5. Accumulation: несколько `backward`, один `step/update` на effective batch.

Сейчас **не обязательно**: gradient penalty, multi-GPU, custom autograd из доки ментора.

Дальше — свой трек (эмбеддинги / NLP / recsys). Карта: [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).
